# 📊 EDA: Berdasarkan Tingkat Disagreement Anotator
Notebook ini menganalisis karakteristik teks dan kualitas data berdasarkan kesepakatan (agreement) antar anotator, sesuai kriteria:
1. **Tanpa Disagreement**: Anotator 1 orang ATAU >1 orang yang setuju 100%.
2. **Disagreement (Tidak 50:50)**: Terdapat perbedaan pendapat, namun ada kelompok mayoritas.
3. **Disagreement (50:50)**: Perbedaan pendapat seimbang sempurna.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import re
from collections import Counter
from wordcloud import WordCloud

# Konfigurasi plot
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('pastel')

df = pd.read_csv('../data/raw/indotoxic2024_annotated_data_v2_final.csv')
print(f"Total baris dataset: {len(df):,}")

### 1. Ekstraksi Kelompok Disagreement
Kita akan mengevaluasi kolom `toxicity` untuk mengklasifikasikan setiap baris ke dalam 3 kelompok utama.

In [ ]:
def determine_disagreement_group(val):
    try:
        votes = ast.literal_eval(val)
        votes = [int(v) for v in votes]
    except:
        return 'Unknown'
    
    n = len(votes)
    if n == 1:
        return 'Tanpa Disagreement'
    
    mean_vote = sum(votes) / n
    if mean_vote == 0.0 or mean_vote == 1.0:
        return 'Tanpa Disagreement'
    elif mean_vote == 0.5:
        return 'Disagreement (50:50)'
    else:
        return 'Disagreement (Tidak 50:50)'

df['agreement_group'] = df['toxicity'].apply(determine_disagreement_group)

# Visualisasi Proporsi Kelompok
group_counts = df['agreement_group'].value_counts()
display(pd.DataFrame(group_counts))

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(group_counts, labels=group_counts.index, autopct='%1.1f%%', startangle=90, colors=['#4CAF50', '#FF9800', '#F44336'])
ax.set_title('Proporsi Kelompok Agreement/Disagreement', fontsize=14, fontweight='bold')
plt.show()

### 2. Data Quality per Kelompok (EDA)
Memeriksa masalah kualitas data (missing value, teks terlalu pendek, atau duplikat) di masing-masing kelompok.

In [ ]:
def check_quality(group_name, data):
    total = len(data)
    missing = data['text'].isnull().sum()
    
    # Hitung jumlah kata
    word_counts = data['text'].fillna('').apply(lambda x: len(str(x).split()))
    short_texts = (word_counts <= 3).sum()
    
    # Deteksi duplikat murni pada kolom teks
    duplicates = data.duplicated(subset=['text']).sum()
    
    return {
        'Kelompok': group_name,
        'Total Data': total,
        'Missing Text': missing,
        'Teks Sangat Pendek (<=3 kata)': short_texts,
        'Teks Duplikat': duplicates
    }

quality_reports = []
for group in df['agreement_group'].unique():
    subset = df[df['agreement_group'] == group]
    quality_reports.append(check_quality(group, subset))

quality_df = pd.DataFrame(quality_reports)
display(quality_df)
print("Interpretasi: Apakah kelompok yang 50:50 memiliki lebih banyak noise/teks pendek yang membuatnya sulit dinilai?")

### 3. Setup Fungsi Word Cloud & Ekstraksi Kata
Kita mendefinisikan *stopwords* (kata hubung) dan fungsi *Word Cloud* agar bisa diaplikasikan secara modular ke tiap kelompok.

In [ ]:
indo_stopwords = set([
    'yang', 'di', 'dan', 'ini', 'itu', 'untuk', 'dengan', 'dari', 'ke', 'pada',
    'dalam', 'adalah', 'juga', 'tidak', 'ada', 'orang', 'yg', 'ya', 'aja',
    'kalo', 'buat', 'sama', 'bisa', 'karena', 'kalau', 'akan', 'aku', 'saya',
    'dia', 'mereka', 'kita', 'kamu', 'udah', 'gak', 'nya', 'kok', 'sih', 'lagi',
    'lebih', 'banyak', 'sudah', 'baru', 'jadi', 'aja', 'ga', 'nggak', 'tak', 'di'
])

def generate_wordcloud(data, title, colormap='Reds'):
    text_corpus = ' '.join(data['text'].dropna().astype(str).str.lower())
    # Hapus URL dan Mention sebelum WordCloud
    text_corpus = re.sub(r'https?://\S+|www\.\S+', '', text_corpus)
    text_corpus = re.sub(r'@\w+', '', text_corpus)
    
    try:
        wc = WordCloud(width=800, height=400, background_color='white', 
                       stopwords=indo_stopwords, colormap=colormap, max_words=100).generate(text_corpus)
        plt.figure(figsize=(10, 5))
        plt.imshow(wc, interpolation='bilinear')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Gagal memuat WordCloud: {e}")
        
def get_top_words(data, top_n=15):
    text_corpus = ' '.join(data['text'].dropna().astype(str).str.lower())
    text_corpus = re.sub(r'https?://\S+|www\.\S+', '', text_corpus)
    text_corpus = re.sub(r'@\w+', '', text_corpus)
    
    tokens = re.findall(r"(?u)\b[\w']+\b", text_corpus)
    tokens = [w for w in tokens if w not in indo_stopwords]
    
    return pd.DataFrame(Counter(tokens).most_common(top_n), columns=['Kata', 'Frekuensi'])

### 4. Analisis Kelompok: Tanpa Disagreement
Kelompok ini berisi teks yang dinilai sangat jelas oleh manusia (baik jelas *toxic* maupun jelas *non-toxic*). Mari kita lihat kata-kata dominan pada bagian yang *Toxic* saja di kelompok ini.

In [ ]:
subset_no_disagree = df[df['agreement_group'] == 'Tanpa Disagreement'].copy()

# Ekstrak label mayoritas (karena tidak ada disagreement, mean vote pasti 0 atau 1)
subset_no_disagree['final_label'] = subset_no_disagree['toxicity'].apply(lambda x: 1 if sum([int(v) for v in ast.literal_eval(x)])/len(ast.literal_eval(x)) > 0.5 else 0)

toxic_no_disagree = subset_no_disagree[subset_no_disagree['final_label'] == 1]

print("Top 15 Kata Dominan (Tanpa Disagreement - Kelas Toxic):")
display(get_top_words(toxic_no_disagree).head(10))

generate_wordcloud(toxic_no_disagree, "Word Cloud: Teks Toxic (Tanpa Disagreement / Konsensus Mutlak)", "Reds")

### 5. Analisis Kelompok: Disagreement (Tidak 50:50)
Kelompok ini memuat perdebatan minoritas vs mayoritas. Mari kita lihat kata dominannya.

In [ ]:
subset_disagree_maj = df[df['agreement_group'] == 'Disagreement (Tidak 50:50)']

print("Top 15 Kata Dominan (Disagreement Mayoritas):")
display(get_top_words(subset_disagree_maj).head(10))

generate_wordcloud(subset_disagree_maj, "Word Cloud: Disagreement (Tidak 50:50)", "Oranges")

### 6. Analisis Kelompok: Disagreement (50:50)
Kelompok ini sangat abu-abu. Anotator terbelah dua sama persis. Apakah didominasi oleh bahasa daerah, sindiran politik, atau sarkasme?

In [ ]:
subset_5050 = df[df['agreement_group'] == 'Disagreement (50:50)']

print("Top 15 Kata Dominan (Disagreement 50:50):")
display(get_top_words(subset_5050).head(10))

generate_wordcloud(subset_5050, "Word Cloud: Ambiguitas Tinggi (Disagreement 50:50)", "Blues")

print("\n=== 10 SAMPEL TEKS DISAGREEMENT 50:50 ===")
# Gunakan try/except untuk mencegah error bila ukuran sampel < 10
n_samples = min(10, len(subset_5050))
for i, text in enumerate(subset_5050['text'].sample(n_samples, random_state=42)):
    print(f"[{i+1}] {text}\n")

### 7. Kesimpulan Karakteristik
* **Tanpa Disagreement:** Biasanya mengandung kata umpatan yang sangat eksplisit dan jelas.
* **Disagreement 50:50:** Biasanya mengandung kalimat abu-abu, menyebut nama tokoh publik (politik), atau kritik yang keras tanpa memuat kata kotor eksplisit, sehingga sangat subyektif.